# Advanced Problems with Solutions: Python Metaclass `__prepare__`

This notebook is an advanced, practice-heavy continuation of the supplied lesson on `__prepare__`.

## Learning goals

You will practice:

- the class-creation lifecycle: `__prepare__` → class body → metaclass `__new__` → metaclass `__init__`
- class keyword arguments
- custom prepared namespaces
- duplicate detection
- declaration-order tracking
- annotation/field collection
- namespace validation
- non-`dict` mapping support
- `__classcell__` correctness
- temporary class-body DSL helpers
- descriptors and `__set_name__`
- `__init_subclass__` vs metaclasses
- dynamic class creation
- cooperative metaclass composition
- class-body name lookup
- a complete declarative schema framework

All examples use only the Python standard library.

> **Best-practice theme:** use `__prepare__` when behavior must happen *while the class body is executing*, or when the namespace itself needs special behavior. Otherwise, prefer simpler hooks such as descriptors, `__init_subclass__`, or class decorators.

## Mental model

For:

```python
class Demo(Base, metaclass=Meta, option=123):
    x = 10
```

the important simplified flow is:

1. Python determines the metaclass.
2. Python calls `Meta.__prepare__("Demo", (Base,), option=123)`.
3. Python executes the class body using the returned object as the local namespace.
4. Python passes the populated namespace to `Meta.__new__`.
5. The metaclass creates the actual class object.
6. Descriptor `__set_name__` and base `__init_subclass__` hooks participate in creation.
7. Metaclass `__init__` finishes initialization.

Modern `dict` objects preserve insertion order. Therefore, use a custom namespace for behavior such as interception, validation, duplicate detection, provenance, or DSL support—not merely to get ordering.

In [1]:
# Shared test helpers

def check(condition, message="check failed"):
    if not condition:
        raise AssertionError(message)

def check_equal(actual, expected, message=None):
    if actual != expected:
        if message is None:
            message = "expected {!r}, got {!r}".format(expected, actual)
        raise AssertionError(message)

print("Test helpers ready.")

Test helpers ready.


# Problem 1 — Trace the class-creation lifecycle

Build a metaclass that records:

- `__prepare__`
- start/end of `__new__`
- `__init__`

Pass `tag="advanced"` in the class header. Seed a value in `__prepare__`, and consume the class keyword rather than blindly forwarding it to `type`.

In [2]:
# Exercise scaffold (safe to run because it is comments only)
#
# events = []
#
# class TraceMeta(type):
#     @classmethod
#     def __prepare__(mcls, name, bases, **kwargs):
#         ...
#
#     def __new__(mcls, name, bases, namespace, **kwargs):
#         ...
#
#     def __init__(cls, name, bases, namespace, **kwargs):
#         ...

## Solution 1

In [3]:
events = []

class TraceMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        events.append(("prepare", name, dict(kwargs)))
        return {"seeded_by_prepare": True}

    def __new__(mcls, name, bases, namespace, **kwargs):
        events.append(("new:start", name, dict(kwargs), list(namespace)))
        tag = kwargs.pop("tag", None)
        cls = super().__new__(mcls, name, bases, namespace)
        cls.tag = tag
        events.append(("new:end", name))
        return cls

    def __init__(cls, name, bases, namespace, **kwargs):
        events.append(("init", name, dict(kwargs)))
        super().__init__(name, bases, namespace)

class Traced(metaclass=TraceMeta, tag="advanced"):
    value = 42

check(Traced.seeded_by_prepare is True)
check_equal(Traced.tag, "advanced")
check_equal(Traced.value, 42)

for event in events:
    print(event)

('prepare', 'Traced', {'tag': 'advanced'})
('new:start', 'Traced', {'tag': 'advanced'}, ['seeded_by_prepare', '__module__', '__qualname__', '__firstlineno__', 'value', '__static_attributes__'])
('new:end', 'Traced')
('init', 'Traced', {'tag': 'advanced'})


### Takeaway

The prepared namespace exists before class-body execution. A class keyword is not automatically a normal class attribute; the metaclass decides how to consume it.

# Problem 2 — Inject configuration in `__prepare__`

Accept `version=` and `feature_flags=` in the class header.

The prepared namespace should expose:

- `__config_version__`
- `__feature_flags__`

Copy the mutable feature dictionary rather than retaining the caller's object.

In [4]:
class ConfigMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        version = kwargs.get("version", 1)
        feature_flags = dict(kwargs.get("feature_flags", {}))
        return {
            "__config_version__": version,
            "__feature_flags__": feature_flags,
        }

    def __new__(mcls, name, bases, namespace, **kwargs):
        kwargs.pop("version", None)
        kwargs.pop("feature_flags", None)
        return super().__new__(mcls, name, bases, namespace)

flags = {"cache": True, "audit": False}

class Service(metaclass=ConfigMeta, version=3, feature_flags=flags):
    endpoint = "/v3/items"

flags["cache"] = False

check_equal(Service.__config_version__, 3)
check_equal(Service.__feature_flags__, {"cache": True, "audit": False})
print(Service.__config_version__, Service.__feature_flags__)

3 {'cache': True, 'audit': False}


# Problem 3 — Log every class-body write

Create a `dict` subclass that records assignments made while the class body executes.

Observe automatically inserted names and user-defined assignments.

In [5]:
class LoggingNamespace(dict):
    def __init__(self):
        super().__init__()
        self.write_log = []

    def __setitem__(self, key, value):
        self.write_log.append((key, type(value).__name__))
        super().__setitem__(key, value)

class LoggingMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return LoggingNamespace()

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace)
        cls.__definition_writes__ = tuple(namespace.write_log)
        return cls

class Logged(metaclass=LoggingMeta):
    answer = 42
    text = "hello"

    def method(self):
        return self.answer

print(*Logged.__definition_writes__, sep="\n")
check(any(name == "answer" for name, _ in Logged.__definition_writes__))
check(any(name == "method" for name, _ in Logged.__definition_writes__))

('__module__', 'str')
('__qualname__', 'str')
('__firstlineno__', 'int')
('answer', 'int')
('text', 'str')
('method', 'function')
('__static_attributes__', 'tuple')


# Problem 4 — Reject duplicate names at definition time

A normal class silently allows:

```python
class Bad:
    timeout = 5
    timeout = 10
```

Create a custom namespace that raises a precise exception on a second assignment to the same class-body key.

In [6]:
class DuplicateNameError(TypeError):
    pass

class NoDuplicateNamespace(dict):
    def __setitem__(self, key, value):
        if key in self:
            raise DuplicateNameError(
                "duplicate class-body name: {!r}".format(key)
            )
        super().__setitem__(key, value)

class NoDuplicateMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return NoDuplicateNamespace()

bad_source = """
class BadConfig(metaclass=NoDuplicateMeta):
    timeout = 5
    retries = 3
    timeout = 10
"""

try:
    exec(bad_source, globals(), globals())
except DuplicateNameError as exc:
    print("Caught as expected:", exc)
else:
    raise AssertionError("duplicate assignment was not rejected")

Caught as expected: duplicate class-body name: 'timeout'


### Why `__prepare__` matters here

After a normal class body finishes, the first value of `timeout` is already gone. A custom namespace can preserve information that would otherwise be lost.

# Problem 5 — Record declaration order

Record user-defined class names in declaration order and store them on the final class as `__declaration_order__`.

Ignore ordinary double-underscore implementation keys.

In [7]:
class RecordingNamespace(dict):
    def __init__(self):
        super().__init__()
        self.user_names = []

    def __setitem__(self, key, value):
        is_dunder = key.startswith("__") and key.endswith("__")
        if not is_dunder and key not in self:
            self.user_names.append(key)
        super().__setitem__(key, value)

class RecordingMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return RecordingNamespace()

    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace)
        cls.__declaration_order__ = tuple(namespace.user_names)
        return cls

class Pipeline(metaclass=RecordingMeta):
    source = "queue"
    workers = 4

    def start(self):
        return "started"

    timeout = 30

check_equal(
    Pipeline.__declaration_order__,
    ("source", "workers", "start", "timeout"),
)
print(Pipeline.__declaration_order__)

('source', 'workers', 'start', 'timeout')


# Problem 6 — Collect annotated fields

Create `__fields__` from annotations.

For each public annotated name, record:

- annotation
- whether a default exists
- the default if present

In [8]:
_MISSING = object()

class FieldCollectingMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        annotations = dict(namespace.get("__annotations__", {}))
        fields = {}

        for field_name, annotation in annotations.items():
            if field_name.startswith("_"):
                continue

            default = namespace.get(field_name, _MISSING)
            fields[field_name] = {
                "annotation": annotation,
                "has_default": default is not _MISSING,
                "default": None if default is _MISSING else default,
            }

        cls = super().__new__(mcls, name, bases, namespace)
        cls.__fields__ = fields
        return cls

class UserRecord(metaclass=FieldCollectingMeta):
    id: int
    name: str = "anonymous"
    active: bool = True
    _internal: str = "skip"

check(set(UserRecord.__fields__) == {"id", "name", "active"})
check(UserRecord.__fields__["id"]["has_default"] is False)
check_equal(UserRecord.__fields__["name"]["default"], "anonymous")

for key, info in UserRecord.__fields__.items():
    print(key, "=>", info)

id => {'annotation': <class 'int'>, 'has_default': False, 'default': None}
name => {'annotation': <class 'str'>, 'has_default': True, 'default': 'anonymous'}
active => {'annotation': <class 'bool'>, 'has_default': True, 'default': True}


### Advanced observation

An annotation-only statement such as `id: int` updates the class namespace's `__annotations__` mapping. It does not necessarily produce a direct `id = ...` write.

# Problem 7 — Enforce reserved-name rules

Reserve names beginning with `framework_`, except for a small allowlist controlled by the framework.

Reject violations during class-body execution.

In [9]:
class ReservedNamespace(dict):
    def __init__(self, allowed=()):
        super().__init__()
        self.allowed = set(allowed)

    def __setitem__(self, key, value):
        if key.startswith("framework_") and key not in self.allowed:
            raise NameError(
                "{!r} is reserved by the framework".format(key)
            )
        super().__setitem__(key, value)

class ReservedMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        ns = ReservedNamespace(allowed={"framework_version"})
        ns["framework_version"] = 1
        return ns

good_source = """
class GoodPlugin(metaclass=ReservedMeta):
    plugin_name = "metrics"
"""

bad_source = """
class BadPlugin(metaclass=ReservedMeta):
    framework_secret = "forbidden"
"""

exec(good_source, globals(), globals())
check_equal(GoodPlugin.framework_version, 1)

try:
    exec(bad_source, globals(), globals())
except NameError as exc:
    print("Caught as expected:", exc)
else:
    raise AssertionError("reserved name was not rejected")

Caught as expected: 'framework_secret' is reserved by the framework


# Problem 8 — Use a non-`dict` prepared namespace safely

Use `collections.UserDict` while the class body executes.

Because `type.__new__` expects a real `dict`, explicitly normalize the mapping before delegating to `super().__new__`.

In [10]:
from collections import UserDict

class UserDictNamespace(UserDict):
    def __init__(self):
        super().__init__()
        self.writes = []

    def __setitem__(self, key, value):
        self.writes.append(key)
        super().__setitem__(key, value)

class MappingMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return UserDictNamespace()

    def __new__(mcls, name, bases, namespace, **kwargs):
        write_log = tuple(namespace.writes)
        normalized = dict(namespace)

        cls = super().__new__(mcls, name, bases, normalized)
        cls.__write_log__ = write_log
        return cls

class Mapped(metaclass=MappingMeta):
    a = 10
    b = 20

    def total(self):
        return self.a + self.b

check_equal(Mapped().total(), 30)
print(Mapped.__write_log__)

('__module__', '__qualname__', '__firstlineno__', 'a', 'b', 'total', '__static_attributes__')


# Problem 9 — Preserve `__classcell__`

When the class body uses zero-argument `super()` or references `__class__`, Python may add `__classcell__` to the namespace.

A metaclass that rebuilds or filters the namespace must preserve it.

First observe a deliberately broken implementation.

In [11]:
class BrokenFilteringMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        # Deliberate bug: removes all dunder keys, including __classcell__.
        filtered = {
            key: value
            for key, value in namespace.items()
            if not (key.startswith("__") and key.endswith("__"))
        }
        return super().__new__(mcls, name, bases, filtered)

broken_source = """
class ParentForBroken:
    def greet(self):
        return "parent"

class BrokenChild(ParentForBroken, metaclass=BrokenFilteringMeta):
    def greet(self):
        return super().greet() + " -> child"
"""

try:
    exec(broken_source, globals(), globals())
except RuntimeError as exc:
    print("Broken metaclass error:", exc)
else:
    try:
        BrokenChild().greet()
    except Exception as exc:
        print("Broken metaclass later error:", type(exc).__name__, exc)
    else:
        raise AssertionError("expected broken __classcell__ handling to fail")

Broken metaclass error: __class__ not set defining 'BrokenChild' as <class '__main__.BrokenChild'>. Was __classcell__ propagated to type.__new__?


## Solution 9 — copy safely

In [12]:
class SafeFilteringMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        # Minimal normalization preserves __classcell__ automatically.
        normalized = dict(namespace)
        return super().__new__(mcls, name, bases, normalized)

class ParentSafe:
    def greet(self):
        return "parent"

class SafeChild(ParentSafe, metaclass=SafeFilteringMeta):
    def greet(self):
        return super().greet() + " -> child"

check_equal(SafeChild().greet(), "parent -> child")
print(SafeChild().greet())

parent -> child


### Best practice

Do not aggressively reconstruct a class namespace. If you must transform it, preserve `__classcell__` and other implementation-required entries.

# Problem 10 — Inject a temporary class-body DSL

Make a helper named `field(...)` available only while the class body executes.

Users should be able to write:

```python
class Product(metaclass=DSLMeta):
    sku = field(str, required=True)
    price = field(float, default=0.0)
```

The final class must not retain the helper itself.

In [13]:
class FieldSpec:
    def __init__(self, python_type, required=False, default=_MISSING):
        self.python_type = python_type
        self.required = required
        self.default = default

    def __repr__(self):
        default = "<missing>" if self.default is _MISSING else repr(self.default)
        return "FieldSpec(type={}, required={}, default={})".format(
            getattr(self.python_type, "__name__", self.python_type),
            self.required,
            default,
        )

def make_field(python_type, required=False, default=_MISSING):
    if required and default is not _MISSING:
        raise ValueError("a required field should not define a default")
    return FieldSpec(python_type, required=required, default=default)

class DSLMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return {"field": make_field}

    def __new__(mcls, name, bases, namespace, **kwargs):
        namespace = dict(namespace)
        namespace.pop("field", None)

        declared = {
            key: value
            for key, value in namespace.items()
            if isinstance(value, FieldSpec)
        }

        cls = super().__new__(mcls, name, bases, namespace)
        cls.__declared_fields__ = declared
        return cls

class Product(metaclass=DSLMeta):
    sku = field(str, required=True)
    price = field(float, default=0.0)
    title = field(str, default="untitled")

check("field" not in Product.__dict__)
check_equal(tuple(Product.__declared_fields__), ("sku", "price", "title"))
print(Product.__declared_fields__)

{'sku': FieldSpec(type=str, required=True, default=<missing>), 'price': FieldSpec(type=float, required=False, default=0.0), 'title': FieldSpec(type=str, required=False, default='untitled')}


# Problem 11 — Trace descriptor `__set_name__`

Show when a descriptor receives `__set_name__` relative to metaclass `__new__`.

In [14]:
descriptor_events = []

class NamedDescriptor:
    def __set_name__(self, owner, name):
        descriptor_events.append(("set_name", owner.__name__, name))
        self.public_name = name

    def __get__(self, instance, owner):
        if instance is None:
            return self
        return instance.__dict__.get(self.public_name)

    def __set__(self, instance, value):
        instance.__dict__[self.public_name] = value

class DescriptorTraceMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        descriptor_events.append(("prepare", name))
        return {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        descriptor_events.append(("new:before-super", name))
        cls = super().__new__(mcls, name, bases, namespace)
        descriptor_events.append(("new:after-super", name))
        return cls

class DescriptorDemo(metaclass=DescriptorTraceMeta):
    value = NamedDescriptor()

for event in descriptor_events:
    print(event)

labels = [item[0] for item in descriptor_events]
check(labels.index("set_name") > labels.index("new:before-super"))
check(labels.index("set_name") < labels.index("new:after-super"))

('prepare', 'DescriptorDemo')
('new:before-super', 'DescriptorDemo')
('set_name', 'DescriptorDemo', 'value')
('new:after-super', 'DescriptorDemo')


# Problem 12 — Compare `__prepare__` with `__init_subclass__`

Trace creation of a subclass.

Use the result to decide which hook is appropriate:

- `__prepare__`: control/intercept the namespace while the body runs.
- `__init_subclass__`: validate/configure an already-created subclass.

In [15]:
hook_events = []

class HookMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        hook_events.append(("prepare", name))
        return {}

    def __new__(mcls, name, bases, namespace, **kwargs):
        hook_events.append(("meta-new-before", name))
        cls = super().__new__(mcls, name, bases, namespace)
        hook_events.append(("meta-new-after", name))
        return cls

class HookBase(metaclass=HookMeta):
    def __init_subclass__(cls, **kwargs):
        hook_events.append(
            ("init_subclass", cls.__name__, tuple(cls.__dict__.keys()))
        )
        super().__init_subclass__(**kwargs)

hook_events.clear()

class HookChild(HookBase):
    alpha = 1
    beta = 2

for event in hook_events:
    print(event)

labels = [event[0] for event in hook_events]
check_equal(labels[0], "prepare")
check("init_subclass" in labels)
check(labels.index("init_subclass") < labels.index("meta-new-after"))

('prepare', 'HookChild')
('meta-new-before', 'HookChild')
('init_subclass', 'HookChild', ('__module__', '__firstlineno__', 'alpha', 'beta', '__static_attributes__', '__doc__'))
('meta-new-after', 'HookChild')


# Problem 13 — Dynamic classes with `types.prepare_class`

Use the standard library to:

1. resolve a metaclass,
2. obtain the prepared namespace,
3. execute class-body code in that namespace,
4. create the class manually.

In [16]:
import types

dynamic_events = []

class DynamicMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        dynamic_events.append(("prepare", name, dict(kwargs)))
        return {"seed": 100}

    def __new__(mcls, name, bases, namespace, **kwargs):
        dynamic_events.append(("new", name, dict(kwargs)))
        multiplier = kwargs.pop("multiplier", 1)
        cls = super().__new__(mcls, name, bases, namespace)
        cls.multiplier = multiplier
        return cls

meta, namespace, remaining_kwds = types.prepare_class(
    "DynamicThing",
    (),
    {"metaclass": DynamicMeta, "multiplier": 5},
)

body = """
value = seed + 23

def compute(self, x):
    return (self.value + x) * self.multiplier
"""

exec(body, globals(), namespace)

DynamicThing = meta(
    "DynamicThing",
    (),
    namespace,
    **remaining_kwds
)

check_equal(DynamicThing.seed, 100)
check_equal(DynamicThing.value, 123)
check_equal(DynamicThing().compute(2), 625)

print(dynamic_events)
print("remaining keywords:", remaining_kwds)

[('prepare', 'DynamicThing', {'multiplier': 5}), ('new', 'DynamicThing', {'multiplier': 5})]
remaining keywords: {'multiplier': 5}


## Bonus — `types.new_class`

In [17]:
def populate_namespace(ns):
    ns["x"] = 7
    ns["square"] = lambda self: self.x ** 2

DynamicTwo = types.new_class(
    "DynamicTwo",
    (),
    {"metaclass": DynamicMeta, "multiplier": 3},
    populate_namespace,
)

check_equal(DynamicTwo().square(), 49)
check_equal(DynamicTwo.multiplier, 3)
print(DynamicTwo, DynamicTwo.x, DynamicTwo.multiplier)

<class '__main__.DynamicTwo'> 7 3


# Problem 14 — Metaclass conflict and cooperative `__prepare__`

Create two unrelated metaclasses, observe the conflict, then combine them through multiple inheritance.

Each metaclass should cooperate through `super()` and add one seed value.

In [18]:
class MetaA(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        ns = super().__prepare__(name, bases, **kwargs)
        ns["from_a"] = "A"
        return ns

class MetaB(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        ns = super().__prepare__(name, bases, **kwargs)
        ns["from_b"] = "B"
        return ns

class BaseA(metaclass=MetaA):
    pass

class BaseB(metaclass=MetaB):
    pass

conflict_source = """
class Conflict(BaseA, BaseB):
    pass
"""

try:
    exec(conflict_source, globals(), globals())
except TypeError as exc:
    print("Metaclass conflict:", exc)
else:
    raise AssertionError("expected a metaclass conflict")

Metaclass conflict: metaclass conflict: the metaclass of a derived class must be a (non-strict) subclass of the metaclasses of all its bases


## Solution 14

In [19]:
class CombinedMeta(MetaA, MetaB):
    pass

class Combined(BaseA, BaseB, metaclass=CombinedMeta):
    own = "C"

check_equal(Combined.from_a, "A")
check_equal(Combined.from_b, "B")
check_equal(Combined.own, "C")

print(Combined.from_a, Combined.from_b, Combined.own)
print("CombinedMeta MRO:", CombinedMeta.__mro__)

A B C
CombinedMeta MRO: (<class '__main__.CombinedMeta'>, <class '__main__.MetaA'>, <class '__main__.MetaB'>, <class 'type'>, <class 'object'>)


### Best practice for composable metaclasses

- use `super()`
- accept `**kwargs`
- consume only the keywords your metaclass layer owns
- avoid hard-coding another metaclass unless you intentionally stop cooperation

# Problem 15 — Class-body name lookup through the prepared namespace

A custom mapping can affect name resolution.

Create a namespace whose `__missing__` shadows exactly one global name while still allowing normal fallback for all other names.

In [20]:
GLOBAL_TOKEN = "from module globals"

class ShadowNamespace(dict):
    def __missing__(self, key):
        if key == "GLOBAL_TOKEN":
            return "shadowed by prepared namespace"
        raise KeyError(key)

class ShadowMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        return ShadowNamespace()

class ShadowDemo(metaclass=ShadowMeta):
    observed = GLOBAL_TOKEN

check_equal(ShadowDemo.observed, "shadowed by prepared namespace")
check_equal(GLOBAL_TOKEN, "from module globals")

print("global:", GLOBAL_TOKEN)
print("class body observed:", ShadowDemo.observed)

global: from module globals
class body observed: shadowed by prepared namespace


### Anti-pattern

Do not return a fake value for every missing key. That can accidentally shadow globals and builtins. Use narrow rules and raise `KeyError` otherwise.

# Problem 16 — Standard-library case study: `Enum`

Inspect the namespace returned by the metaclass used by `Enum`.

In [21]:
from enum import Enum

enum_meta = type(Enum)
prepared_for_enum = enum_meta.__prepare__("TrafficLight", (Enum,))

print("Enum metaclass:", enum_meta)
print("Prepared namespace type:", type(prepared_for_enum))

class TrafficLight(Enum):
    RED = 1
    YELLOW = 2
    GREEN = 3

check_equal(
    [member.name for member in TrafficLight],
    ["RED", "YELLOW", "GREEN"],
)
print(list(TrafficLight))

Enum metaclass: <class 'enum.EnumType'>
Prepared namespace type: <class 'enum.EnumDict'>
[<TrafficLight.RED: 1>, <TrafficLight.YELLOW: 2>, <TrafficLight.GREEN: 3>]


# Problem 17 — Class decorator vs `__prepare__`

A class decorator runs after the class exists.

Show that it can add metadata, but it cannot recover an overwritten value from an earlier duplicate assignment in the class body.

In [22]:
def tagged_class(tag):
    def decorate(cls):
        cls.decorator_tag = tag
        return cls
    return decorate

@tagged_class("post-creation")
class Decorated:
    x = 1
    x = 2

check_equal(Decorated.x, 2)
check_equal(Decorated.decorator_tag, "post-creation")

print("Decorator sees final x =", Decorated.x)
print("A custom prepared namespace could have detected the duplicate.")

Decorator sees final x = 2
A custom prepared namespace could have detected the duplicate.


# Problem 18 — Capstone: a declarative schema framework

Build:

```python
class Account(Schema):
    username = field(str, required=True)
    age = field(int, default=0)
```

Requirements:

1. `field` exists only during class-body execution.
2. Duplicate class-body names are rejected.
3. Field order is preserved.
4. Fields are inherited.
5. Subclasses can override inherited fields.
6. Required fields are enforced.
7. Runtime types are validated.
8. Unknown constructor arguments are rejected.
9. Defaults are applied.
10. `to_dict()` returns clean instance data.
11. The metaclass remains compatible with zero-argument `super()`.

In [23]:
# Exercise scaffold
#
# class SchemaNamespace(dict):
#     ...
#
# class SchemaField:
#     ...
#
# class SchemaMeta(type):
#     ...
#
# class Schema(metaclass=SchemaMeta):
#     ...

## Solution 18 — full implementation

In [24]:
class SchemaField:
    def __init__(self, python_type, required=False, default=_MISSING):
        if required and default is not _MISSING:
            raise ValueError("required field cannot define a default")

        self.python_type = python_type
        self.required = required
        self.default = default
        self.name = None

    def __set_name__(self, owner, name):
        self.name = name

    def validate(self, value):
        if not isinstance(value, self.python_type):
            raise TypeError(
                "{} must be {}, got {}".format(
                    self.name,
                    getattr(self.python_type, "__name__", self.python_type),
                    type(value).__name__,
                )
            )

    def __get__(self, instance, owner):
        if instance is None:
            return self

        if self.name in instance.__dict__:
            return instance.__dict__[self.name]

        if self.default is not _MISSING:
            return self.default

        raise AttributeError(self.name)

    def __set__(self, instance, value):
        self.validate(value)
        instance.__dict__[self.name] = value

    def __repr__(self):
        return "SchemaField(name={!r}, type={}, required={})".format(
            self.name,
            getattr(self.python_type, "__name__", self.python_type),
            self.required,
        )

def schema_field(python_type, required=False, default=_MISSING):
    return SchemaField(
        python_type,
        required=required,
        default=default,
    )

class SchemaNamespace(dict):
    def __init__(self):
        super().__init__()
        self.declared_names = []

    def __setitem__(self, key, value):
        if key in self and key != "__annotations__":
            raise DuplicateNameError(
                "duplicate class-body name in schema: {!r}".format(key)
            )

        if isinstance(value, SchemaField):
            self.declared_names.append(key)

        super().__setitem__(key, value)

class SchemaMeta(type):
    @classmethod
    def __prepare__(mcls, name, bases, **kwargs):
        ns = SchemaNamespace()
        # Seed the helper without making it look like a user declaration.
        dict.__setitem__(ns, "field", schema_field)
        return ns

    def __new__(mcls, name, bases, namespace, **kwargs):
        own_fields = {
            key: value
            for key, value in namespace.items()
            if isinstance(value, SchemaField)
        }

        normalized = dict(namespace)
        normalized.pop("field", None)

        # dict(namespace) preserves __classcell__ if Python created one.
        cls = super().__new__(mcls, name, bases, normalized)

        inherited = {}
        for base in reversed(cls.__mro__[1:]):
            inherited.update(getattr(base, "__schema_fields__", {}))

        inherited.update(own_fields)
        cls.__schema_fields__ = inherited
        cls.__schema_field_order__ = tuple(inherited)
        return cls

class Schema(metaclass=SchemaMeta):
    def __init__(self, **values):
        fields = self.__schema_fields__

        unknown = set(values) - set(fields)
        if unknown:
            raise TypeError(
                "unknown field(s): {}".format(", ".join(sorted(unknown)))
            )

        for name, spec in fields.items():
            if name in values:
                setattr(self, name, values[name])
            elif spec.default is not _MISSING:
                setattr(self, name, spec.default)
            elif spec.required:
                raise TypeError("missing required field: {}".format(name))

    def to_dict(self):
        result = {}
        for name in self.__schema_fields__:
            try:
                result[name] = getattr(self, name)
            except AttributeError:
                result[name] = None
        return result

In [25]:
class Account(Schema):
    username = field(str, required=True)
    age = field(int, default=0)

    def describe(self):
        return "{}:{}".format(self.username, self.age)

class PremiumAccount(Account):
    tier = field(str, default="gold")
    age = field(int, default=18)

account = Account(username="alice")
premium = PremiumAccount(username="bob", tier="platinum")

check_equal(account.to_dict(), {"username": "alice", "age": 0})
check_equal(
    premium.to_dict(),
    {"username": "bob", "age": 18, "tier": "platinum"},
)
check_equal(
    PremiumAccount.__schema_field_order__,
    ("username", "age", "tier"),
)
check("field" not in Account.__dict__)
check("field" not in PremiumAccount.__dict__)

print(account.to_dict())
print(premium.to_dict())
print(PremiumAccount.__schema_fields__)

{'username': 'alice', 'age': 0}
{'username': 'bob', 'age': 18, 'tier': 'platinum'}
{'username': SchemaField(name='username', type=str, required=True), 'age': SchemaField(name='age', type=int, required=False), 'tier': SchemaField(name='tier', type=str, required=False)}


In [26]:
# Prove zero-argument super() still works with the schema metaclass.

class LabelSchema(Schema):
    def label(self):
        return "base"

class ChildLabelSchema(LabelSchema):
    def label(self):
        return super().label() + "->child"

check_equal(ChildLabelSchema().label(), "base->child")
print(ChildLabelSchema().label())

base->child


In [27]:
# Capstone failure tests

def expect_error(exc_type, fn, contains=None):
    try:
        fn()
    except exc_type as exc:
        if contains is not None:
            check(
                contains in str(exc),
                "error message did not contain {!r}".format(contains),
            )
        print("Expected {}: {}".format(exc_type.__name__, exc))
    else:
        raise AssertionError("expected {}".format(exc_type.__name__))

expect_error(
    TypeError,
    lambda: Account(),
    "missing required field",
)

expect_error(
    TypeError,
    lambda: Account(username="alice", unknown=1),
    "unknown field",
)

expect_error(
    TypeError,
    lambda: Account(username=123),
    "username must be str",
)

duplicate_schema_source = """
class BadSchema(Schema):
    x = field(int, default=1)
    x = field(int, default=2)
"""

expect_error(
    DuplicateNameError,
    lambda: exec(duplicate_schema_source, globals(), globals()),
    "duplicate class-body name",
)

Expected TypeError: missing required field: username
Expected TypeError: unknown field(s): unknown
Expected TypeError: username must be str, got int
Expected DuplicateNameError: duplicate class-body name in schema: 'x'


# Problem 19 — Add computed framework metadata

Extend `SchemaMeta` cooperatively.

Create a human-readable signature:

```text
name:str(required), retries:int(default=3)
```

Do the work after the base schema metaclass has built `__schema_fields__`.

In [28]:
class DescribedSchemaMeta(SchemaMeta):
    def __new__(mcls, name, bases, namespace, **kwargs):
        cls = super().__new__(mcls, name, bases, namespace, **kwargs)

        parts = []

        for field_name, spec in cls.__schema_fields__.items():
            type_name = getattr(
                spec.python_type,
                "__name__",
                str(spec.python_type),
            )

            if spec.required:
                status = "required"
            elif spec.default is not _MISSING:
                status = "default={!r}".format(spec.default)
            else:
                status = "optional"

            parts.append(
                "{}:{}({})".format(field_name, type_name, status)
            )

        cls.__schema_signature__ = ", ".join(parts)
        return cls

class DescribedSchema(Schema, metaclass=DescribedSchemaMeta):
    pass

class Event(DescribedSchema):
    name = field(str, required=True)
    retries = field(int, default=3)

print(Event.__schema_signature__)
check_equal(
    Event.__schema_signature__,
    "name:str(required), retries:int(default=3)",
)

name:str(required), retries:int(default=3)


# Problem 20 — Choose the simplest hook

Use this decision table as a review exercise.

| Requirement | Usually prefer |
|---|---|
| Detect duplicate assignments while the body runs | custom `__prepare__` namespace |
| Add metadata after a class exists | class decorator |
| Configure/validate subclasses | `__init_subclass__` |
| React when a descriptor gets a name | descriptor `__set_name__` |
| Control actual class-object creation | metaclass `__new__` |
| Provide temporary DSL names in the class body | `__prepare__` |
| Preserve ordinary declaration order only | ordinary modern `dict` |

The advanced skill is not just writing metaclasses. It is choosing the least-complex mechanism that correctly observes the information you need.

# Extra challenge set

Try these after completing the solved problems:

1. Allow duplicate assignments only for names in an explicit override allowlist.
2. Record source line numbers for class-body assignments and discuss reliability trade-offs.
3. Add aliases to the schema DSL.
4. Add a `strict=False` class keyword consumed cooperatively by the metaclass.
5. Compose two independent namespace features through `super()`.
6. Build a namespace that records reads as well as writes.
7. Reject public class attributes that are not snake_case.
8. Support abstract schema fields inherited from a base class.
9. Rebuild the schema framework with `__init_subclass__` and compare limitations.
10. Generate schema classes at runtime with `types.new_class`.
11. Add field validators such as min/max or regex.
12. Add serializer hooks per field.
13. Add immutable/frozen schema instances.
14. Add a metaclass option that forbids field overrides.
15. Add a namespace that explains exactly which class-body write violated a rule.

# Best-practices checklist

- Keep `__prepare__` small and deterministic.
- Return a writable namespace suitable for class-body execution.
- If the namespace is not a `dict`, normalize it before calling `type.__new__`.
- Preserve `__classcell__` when copying/filtering.
- Use cooperative `super()`.
- Accept `**kwargs`; consume only the keywords your layer owns.
- Do not blindly forward arbitrary class keywords to `type`.
- Prefer narrow allowlists over magical `__missing__` behavior.
- Use normal `dict` when custom interception is unnecessary.
- Prefer descriptors, `__init_subclass__`, or decorators when they solve the problem more simply.
- Test both successful class creation and intentional definition-time failures.
- Keep framework error messages precise.